# Multi-Agent Orchestration Patterns

## Why this matters

The X++ code review agent runs four specialist agents (Security, Performance, Best Practice, Naming) concurrently via `ThreadPoolExecutor` and merges their results. That's a real, working instance of multi-agent orchestration — but it's the simplest possible pattern (parallel, independent, no cross-talk). This notebook covers the full spectrum of orchestration patterns, where the X++ project's design sits on that spectrum, and what would need to change if the agents needed to depend on each other instead of running independently.

## Level 1: What it is

"Multi-agent" just means more than one LLM call/role is involved in producing a result, coordinated by some orchestration logic outside the model itself. The orchestration logic is regular code (Python, in our case) — the model doesn't orchestrate itself unless you deliberately build that in (see the supervisor pattern below). Four common patterns, roughly in order of increasing coordination complexity:

1. **Parallel / fan-out-fan-in** — N independent agents run concurrently on the same input, results are merged at the end. No agent knows about the others. **This is the X++ project's pattern.**
2. **Sequential / pipeline** — agent A's output becomes agent B's input, in a fixed order. Each stage transforms or narrows the work.
3. **Supervisor / orchestrator-worker** — a coordinating agent (itself an LLM call) decides *at runtime* which worker agent(s) to invoke, in what order, and whether to invoke them again based on results.
4. **Swarm / peer-to-peer** — agents can hand off to each other directly, no central coordinator; control passes between peers based on each agent's own decision about who should act next.

## Level 2: How it actually works internally

### Parallel / fan-out-fan-in (the X++ pattern)

```python
with ThreadPoolExecutor(max_workers=len(agents_to_run)) as executor:
    future_to_name = {executor.submit(agent_fn, xpp): name for name, agent_fn in agents_to_run}
    for future in as_completed(future_to_name):
        issues = future.result()
        all_issues.extend(issues)
```

Why threads work here despite Python's GIL: each `agent_fn` call is dominated by waiting on a network response from the Claude API. The GIL is released during that I/O wait, so four threads genuinely overlap their waiting time — this is **concurrency for I/O-bound work**, not true parallel computation. If the agents were doing heavy local computation (not API calls), threads wouldn't help due to the GIL, and you'd need `multiprocessing` instead.

The defining trait: **agents don't communicate.** Security has zero awareness that Performance is also running, let alone what it found. This makes the pattern trivially parallelizable and easy to reason about, but limits it to problems where each sub-task is genuinely independent.

### Sequential / pipeline

```python
extracted = extraction_agent(document)
summary = summarization_agent(extracted)
final = formatting_agent(summary)
```

Each stage's *entire output* becomes the next stage's *entire input*. This is simple to build but has a specific failure mode worth naming: **error propagation without visibility.** If `extraction_agent` silently drops a field, `summarization_agent` never sees it and has no way to know something is missing — it just summarizes what it was given, confidently. Compare to the X++ project's `AgentCallError` design: each agent's failure is caught and surfaced independently, rather than one agent's bad output silently degrading the next one's quality.

### Supervisor / orchestrator-worker

The supervisor is itself an LLM call, typically using tool-use, where each "tool" is actually a call to a different worker agent:

```python
SUPERVISOR_TOOLS = [
    {"name": "run_security_review", "description": "..."},
    {"name": "run_performance_review", "description": "..."},
    {"name": "request_more_context", "description": "..."},
]
# supervisor decides which to call, possibly iterating:
# e.g. "Security found a possible SQL injection -- let me also run
# Performance on this same method to see if there's a related N+1 issue"
```

This is genuinely different from parallel fan-out: the *decision* of which agents to run, and whether to re-run one with refined instructions, is made by a model at runtime, not fixed in code ahead of time. This is what you'd need if, say, you wanted the review pipeline to decide "this class is unusually large, spend extra passes on Security" — the X++ project's current design can't do that; every agent always runs (if its checkbox is on), regardless of what any other agent finds.

### Swarm / peer-to-peer

Each agent, after acting, can explicitly hand off control to a named peer instead of returning to a central coordinator:

```python
# Agent A's tool schema includes a "handoff_to" option
# If Agent A decides Agent B is better suited to continue,
# it calls handoff_to("agent_b", context=...) instead of finishing
```

Used when the *set* of relevant agents and the *order* they should act in is genuinely unpredictable ahead of time — e.g. a customer support system where a billing agent might discover it's actually a technical issue and hand off to a technical agent, which might hand back to billing. Rare in practice compared to supervisor patterns, because peer-to-peer handoff logic embedded in *every* agent's own prompt is harder to reason about and debug than one central supervisor's decisions.

## Level 3: Where it breaks, and what it doesn't solve

1. **Parallel pattern: no cross-agent context means duplicate/overlapping findings.** This is a real thing you already saw — Security and Best Practice both flagging the same SQL injection from different angles, Performance and Best Practice both flagging the same loop-select. Nothing about `ThreadPoolExecutor` prevents this, because the agents genuinely don't know about each other. Fixing this requires either (a) a dedup/merge pass after fan-in, or (b) moving to a supervisor pattern where the coordinator can see all results before deciding whether a finding is redundant.

2. **Sequential pattern: total latency is additive, not overlapped.** If each stage takes 5 seconds, a 3-stage pipeline takes 15 seconds minimum, no matter how you parallelize within a stage — the stages themselves are inherently serial because each depends on the previous one's output.

3. **Supervisor pattern: the supervisor itself is a single point of failure AND a single point of cost.** Every worker invocation now costs an *additional* LLM call (the supervisor's own reasoning about what to do), on top of the worker calls themselves. For a fixed, known set of checks (like the X++ project's four agents, which should always all run), a supervisor adds cost and latency for a decision that doesn't actually need to be dynamic — this is why the X++ project's parallel pattern is the *right* choice for it, not a limitation to fix. Use a supervisor only when the set/order of agents genuinely can't be decided ahead of time.

4. **Swarm pattern: debugging is genuinely harder.** When any agent can hand off to any other agent based on its own judgment, tracing *why* a particular agent ended up handling a given request requires reconstructing the full handoff chain after the fact — there's no single place in the code where "the decision" was made, unlike a supervisor's centralized tool-call log.

5. **None of these patterns solve non-determinism.** Orchestration is about *coordination*, not about making any individual agent's judgment more consistent. The Security/Performance agents' remaining LLM-judgment variance (discussed in the X++ project sessions) exists regardless of which orchestration pattern wraps them -- that's a property of the individual LLM call, not the coordination layer around it.

6. **Cost compounds fast with nested patterns.** A supervisor that calls 4 workers, each of which is itself a 2-stage pipeline, is 1 (supervisor) + 4×2 (workers) = 9 LLM calls for what might have been answerable in 1-2 calls with a simpler design. Orchestration complexity should be justified by a genuine coordination need, not added because it sounds more sophisticated.

In [ ]:
# Minimal supervisor pattern sketch -- for comparison against the X++
# project's fixed parallel fan-out. NOT what the X++ project uses; this
# is here to make the contrast concrete.

from anthropic import Anthropic

client = Anthropic()

SUPERVISOR_TOOLS = [
    {
        "name": "run_security_review",
        "description": "Run the security specialist agent on the code.",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "run_performance_review",
        "description": "Run the performance specialist agent on the code.",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "finish",
        "description": "Call when no further agents need to run.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

# In a real supervisor loop, you'd call the model, inspect which tool(s)
# it chose, actually invoke the corresponding worker agent function,
# feed the result back as a tool_result message, and repeat until the
# model calls "finish". This is fundamentally a while-loop around
# messages.create(), not a single call -- unlike the X++ project's
# fixed fan-out, the NUMBER of LLM calls here isn't known in advance.

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    tools=SUPERVISOR_TOOLS,
    messages=[{
        "role": "user",
        "content": "This class only does file I/O, no database access at all. "
                    "Decide which review agents are worth running.",
    }],
)

for block in response.content:
    if block.type == "tool_use":
        print(f"Supervisor chose: {block.name}")
# A well-designed supervisor might skip run_security_review here if it
# reasons that file-I/O-only code has a different risk profile than
# database-writing code -- a decision the X++ project's fixed fan-out
# can't make, because every checked agent always runs regardless of
# what the code actually contains.

## Interview-ready summary

**Q: "Walk me through how you'd design a multi-agent system."**

Weak answer: "I'd have a bunch of agents and have them talk to each other."

Strong answer: "It depends entirely on whether the set of sub-tasks is fixed and independent, or needs runtime decisions. In the X++ code review project, I have four specialist agents -- Security, Performance, Best Practice, Naming -- that always all need to run and don't depend on each other's output, so I used a parallel fan-out with `ThreadPoolExecutor`: real concurrency since the work is I/O-bound (waiting on API calls), not CPU-bound, so the GIL isn't a bottleneck. If instead the set of relevant checks depended on what the code actually contained -- skip Security entirely for a class with no database access, for instance -- that would call for a supervisor pattern instead, where a coordinating LLM call decides which workers to invoke. I'd default to the simplest pattern that fits the actual coordination need, since a supervisor adds real cost and latency for decisions that a fixed pipeline doesn't need to make."